## Stage 3.1

In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
%load_ext autoreload
%autoreload 2

import gc
import os
import sys
import gymnasium as gym
import torch

# Limit root process PyTorch threads to prevent CPU spikes
torch.set_num_threads(1)
gc.collect()

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import LeagueHaxballEnv, LeagueOpponentController
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import FixedMultiAgentReset
from src.rl.reward_shapers import Stage3SelfPlayReward
from src.rl.trainer import train_ppo_league

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

POOL_DIR = "models/stage3_league/pool"
os.makedirs(POOL_DIR, exist_ok=True)
OBS_DIM = 84
NUM_ENVS = 16


def make_league_env():
    roster = [
        PlayerSlot(
            team="red",
            stats=PlayerStats(name="RL_Agent", accel=3200.0),
            controller="RL",
            role="ST",
        ),
        PlayerSlot(
            team="blue",
            stats=PlayerStats(name="League_Bot", accel=3200.0),
            controller=LeagueOpponentController(
                pool_dir=POOL_DIR, device="cpu", obs_dim=OBS_DIM
            ),
            role="ST",
        ),
    ]

    match_cfg = MatchConfig(
        mode=ClassicMatchMode(time_limit=10.0, score_limit=1),
        roster=roster,
        time_limit=10.0,
        score_limit=1,
    )

    return LeagueHaxballEnv(
        match_config=match_cfg,
        reward_shaper=Stage3SelfPlayReward(),
        reset_strategy=FixedMultiAgentReset(offset_x=180.0),
        max_steps=600,
    )
# Single-process vectorized execution (Drops RAM from ~25 GB to ~1.5 GB)
train_envs = gym.vector.SyncVectorEnv([make_league_env for _ in range(NUM_ENVS)])
eval_env = make_league_env()

# ─────────────────────────────────────────────────────────────
# Graft Checkpoint (80 dims -> 84 dims)
# ─────────────────────────────────────────────────────────────
model = ActorCritic(obs_dim=OBS_DIM).to(device)


train_ppo_league(
    envs=train_envs,
    eval_env=eval_env,
    model=model,
    device=device,
    total_timesteps=300_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=20,
    save_dir="models/stage3_league",
    pool_dir=POOL_DIR,
    lr_initial=3e-4,
    lr_final=1e-5,
    ent_coef_initial=0.05,
    ent_coef_final=0.001,
)

train_envs.close()
eval_env.close()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
⚡ Device: cuda
🚀League Training Started | Agents/Env: 1 | Dim: 84 | Batch Size: 4096

📊 [STRICT EVAL @ Step  102400]
   Vs Prev Best: 0W - 0L - 10D
   Vs Heuristic: 0W - 8L - 2D
   ❌ FAILED PROMOTION. Retaining previous best model.

📊 [STRICT EVAL @ Step  200704]
   Vs Prev Best: 0W - 0L - 10D
   Vs Heuristic: 0W - 8L - 2D
   ❌ FAILED PROMOTION. Retaining previous best model.


KeyboardInterrupt: 

## Stage 3.2

In [16]:
%load_ext autoreload
%autoreload 2

import os
import sys
import gc
import gymnasium as gym
import torch

# Force single-threaded CPU execution to prevent thread bloat
torch.set_num_threads(1)
gc.collect()

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import LeagueOpponentController, LeagueHaxballEnv
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import TwoVTwoScrambleReset
from src.rl.reward_shapers import Stage3_2RoleReward
from src.rl.trainer import train_ppo_league

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

POOL_DIR = "models/stage3_2_roles/pool"
os.makedirs(POOL_DIR, exist_ok=True)
OBS_DIM = 84
NUM_ENVS = 16


def make_2v2_env():
    # Red Team: Active Learners
    red_roster = [
        PlayerSlot("red", PlayerStats("Red_ST", accel=3200.0), controller="RL", role="ST"),
        PlayerSlot("red", PlayerStats("Red_CB", accel=3200.0), controller="RL", role="CB"),
    ]

    # Blue Team: League Opponents
    blue_roster = [
        PlayerSlot(
            "blue",
            PlayerStats("Blue_ST", accel=3200.0),
            controller=LeagueOpponentController(POOL_DIR, device="cpu", obs_dim=OBS_DIM),
            role="ST",
        ),
        PlayerSlot(
            "blue",
            PlayerStats("Blue_CB", accel=3200.0),
            controller=LeagueOpponentController(POOL_DIR, device="cpu", obs_dim=OBS_DIM),
            role="CB",
        ),
    ]

    match_cfg = MatchConfig(
        mode=ClassicMatchMode(time_limit=15.0, score_limit=1),
        roster=red_roster + blue_roster,
        time_limit=15.0,
        score_limit=1,
    )

    return LeagueHaxballEnv(
        match_config=match_cfg,
        reward_shaper=Stage3_2RoleReward(),
        reset_strategy=TwoVTwoScrambleReset(),
        max_steps=900,
    )




# SyncVectorEnv runs in the main process: 0 duplicated runtimes, ~1.5 GB RAM total
train_envs = gym.vector.SyncVectorEnv([make_2v2_env for _ in range(NUM_ENVS)])
eval_env = make_2v2_env()

# ─────────────────────────────────────────────────────────────
# Surgical Checkpoint Grafting (80 dims -> 84 dims)
# ─────────────────────────────────────────────────────────────
model = ActorCritic(obs_dim=OBS_DIM).to(device)


base_checkpoint = "models/stage1/best_model.pt"
old_state_dict = torch.load(base_checkpoint, map_location=device, weights_only=False)
new_state_dict = model.state_dict()

# Graft weights into 84-dim network
for name, param in old_state_dict.items():
    if "shared.0.weight" in name or "actor_net.0.weight" in name or "critic_net.0.weight" in name:
        new_state_dict[name][:, :80] = param
    else:
        if name in new_state_dict and new_state_dict[name].shape == param.shape:
            new_state_dict[name] = param


state_dict = torch.load("models/stage3_league/best_model.pt", map_location=device, weights_only=False)


model.load_state_dict(new_state_dict)
print(f"✅ Grafted base checkpoint into 84-dim network.")

train_ppo_league(
    envs=train_envs,
    eval_env=eval_env,
    model=model,
    device=device,
    total_timesteps=300_000,
    num_envs=NUM_ENVS,
    num_steps=512,
    eval_freq=100_000,
    eval_episodes=20,
    save_dir="models/stage3_2_roles",
    pool_dir=POOL_DIR,
    lr_initial=8e-5,
    lr_final=8e-6,
    ent_coef_initial=0.005,
    ent_coef_final=0.0002,
)

train_envs.close()
eval_env.close()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
⚡ Device: cuda
✅ Grafted base checkpoint into 84-dim network.
🚀League Training Started | Agents/Env: 2 | Dim: 84 | Batch Size: 16384

📊 [STRICT EVAL @ Step  114688]
   Vs Prev Best: 2W - 2L - 6D
   Vs Heuristic: 2W - 3L - 5D
   ❌ FAILED PROMOTION. Retaining previous best model.

📊 [STRICT EVAL @ Step  212992]
   Vs Prev Best: 2W - 2L - 6D
   Vs Heuristic: 0W - 4L - 6D
   ❌ FAILED PROMOTION. Retaining previous best model.

📊 [STRICT EVAL @ Step  311296]
   Vs Prev Best: 1W - 5L - 4D
   Vs Heuristic: 2W - 2L - 6D
   ❌ FAILED PROMOTION. Retaining previous best model.


# Test

In [7]:
import torch
from src.rl.ppo_core import ActorCritic
from src.rl.benchmarker import RLController, run_arena, run_solo_drill, render_match
from config.match_config import PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController

device = torch.device("cpu") # Fast inference on CPU

# 1. Load RL models
obs_dim = 84
stage2_model = ActorCritic(obs_dim).to(device)
stage2_model.load_state_dict(torch.load("models/stage3_league/best_model.pt", map_location=device))


# 2. Setup Team Coordinators
red_rl_controller = RLController(stage2_model, team="red", device=device)

red_heuristic_coord = TeamHeuristicCoordinator(team="red")
red_heuristic_controller = HeuristicBotController(red_heuristic_coord)

blue_heuristic_coord = TeamHeuristicCoordinator(team="blue")
blue_heuristic_controller = HeuristicBotController(blue_heuristic_coord)

blue_rl_controller = RLController(stage2_model, team="blue")



In [8]:
# ==========================================
# TEST 1: The Diagnostic Solo Drill
# ==========================================
print("--- TEST 1: EMPTY NET DIAGNOSTIC ---")
run_solo_drill(
    agent_roster=[PlayerSlot("red", PlayerStats("RL_Test"), red_rl_controller)],
    num_episodes=5,
    time_limit=60.0
)


--- TEST 1: EMPTY NET DIAGNOSTIC ---
🎯 Running Solo Drill: 60.0s per episode (5 Episodes)
   Episode 1: 0 goals
   Episode 2: 0 goals
   Episode 3: 0 goals
   Episode 4: 0 goals
   Episode 5: 2 goals
📊 Average Scoring Rate: 0.40 goals / 60.0s



0.4

In [11]:
# ==========================================
# TEST 2: The Arena 
# RL Agent (Red) vs Heuristic Bot (Blue)
# ==========================================
print("\n--- TEST 2: THE ARENA (1v1) ---")
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]

stats = run_arena(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=20,
    time_limit=60.0,  # 60 second matches
    score_limit=3
)




--- TEST 2: THE ARENA (1v1) ---
🏟️ Running Arena: 1 RED vs 1 BLUE (20 Matches)
✅ Completed in 80.13s
🏆 Series Outcome (Wins): RED 11 | BLUE 6 | DRAWS 3
⚽ Avg Goals / Match:     RED 1.90 | BLUE 1.55



In [10]:
# ==========================================
# TEST 3: Kaggle-Style Visualization
# Watch the matchup in HTML format
# ==========================================
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]


print("\n--- TEST 3: RENDER MATCH ---")
render_match(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=90.0,
    save_path="renders/arena"
)



--- TEST 3: RENDER MATCH ---
🎬 Generating 1 replays...
Game 1 Result: RED WINS! 🎉 (3 - 0)
Replay saved to: renders/arena/2026-08-24_03-03-17_match_1.html



In [11]:
from src.rl.benchmarker import render_solo_drill

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
agent_slot = PlayerSlot("red", PlayerStats("RL_Agent"), RLController(model, team="red", device=device))

# Render 3 randomized episodes (20 seconds each)
replay_path = render_solo_drill(
    agent_slot=agent_slot,
    num_episodes=3,
    time_limit=20.0,
    save_path="renders/solo_drills",
)

🎬 Generating 3 Solo Drill Replays (20.0s each)...
   Episode 1 Finished: 0 Goals Scored
   Episode 2 Finished: 0 Goals Scored
   Episode 3 Finished: 0 Goals Scored
🏆 Overall: 0.00 Avg Goals / 20.0s
Replay saved to: renders/solo_drills/2026-08-23_07-35-33_solo_drill.html

